# Soup cached128 + frozen tied-weight storage — direct GPU queue screen

The installed solution is identical to the prior fixed experiment and retains the real original Triton kernels. A fixed settings change enables lossless frozen tied-weight storage before training. This development notebook omits an optional readiness preflight that failed before training on the service image; it does not change, bypass or replace the original task/harness or report exports. The three supplied template cells are unchanged. Judge comparisons remain separate from this single public-data run.


In [ ]:
import os, sys, json, subprocess, shutil
from pathlib import Path
repo = Path(os.environ['SOUP_REPO'])
results = Path(os.environ['SOUP_RESULTS'])
print('Editable challenge repo:', repo)
print('Public data:', list((repo / 'data').glob('*.jsonl')))
print('Write your solution files under', repo / 'solution')
# Add notebook cells here to replace solution/__init__.py and your kernels.
# The prepared starter is retained when you run this template unchanged.


In [ ]:
import base64, hashlib, io, stat, zipfile
from pathlib import PurePosixPath

# No imports of Torch/CUDA here. The training subprocess imports solution.
FUSION_SOLUTION_SHA256 = 'dfe2b63ee35ff1e93dd2fa109a7eed84b0ac0f45cde50a0b4c4e9f89c8aee8e2'
FUSION_ARCHIVE_SHA256 = '8eace42af169747d621a7b274424a873ecf08c53000ead5fbdab43ea59214bc2'
FUSION_FILES = {'solution/__init__.py': 'd60862efc1dc58d943ee0f8026d81da9700c8ed6a025c266e0ca7c68ad782cb3', 'solution/activation_storage.py': '95f66325e0361ced6e0c56ad06a027c66ac8d371969dc44a5840f746dfd21ca8', 'solution/cached_head.py': '8517f37d7d879b35b8b4255460f9a6bf39fe57c05375a69cd5c8d0e800c14de6', 'solution/frozen_storage.py': 'c27dc833f9b7ceb15b3f7e96169c5a6572060abead71c093aac8625636aa0fe0', 'solution/head_fused_t4.py': 'ac75f93ce5b46eeb564afd8c74409cbbeda36804c0399d639c159bf8779b14ce', 'solution/head_matmul.py': 'c834bc001e3c6f9449ef4f5cd60d34aa961262e27c81e98e9a47186af5ec7e39', 'solution/head_matmul_t4.py': 'b29ed64e0d1537f6682d2b75d5a979c8269030a306aa8f1dee263bb9e3be3134', 'solution/integration.py': 'fd2db3fc3f367751f9eb439fa5901dbf16a06934e4d6ae56fd61063b4363cb5b', 'solution/lora_dropout.py': 'c6f4df8cc95eac7a76b36b0a84c0372b2678192cabe7d4227302378587bdd277', 'solution/lora_epilogue.py': '4fbbd8697ce95283458e81b6afa7aca581b9604eef7ae5f4565d7b38d2ac6f2f', 'solution/nf4.py': '9e07a73d45f7b7119fc1f5d1d3422351e5d34e2fbd9612f7b90f56bc17ad697a', 'solution/rmsnorm.py': 'bf2c73ff515745e0d071890e011f1fd926fcb1b3415d1c978b56bf51bd47ed5a', 'solution/rotary.py': '8e8ef322d70eba6ac898a040a14d74b9dc8e276b3f93bd033414eeaf927b5429', 'solution/settings.py': 'a77ff94cfae7b5b7ce1474ae260537988196f1491b4bfe113cc93b3d8d47c253', 'solution/shared_attention_mask.py': '93b6f6c2704685075fb14ac29901445350e0cf1c89a453ee1ef6e9500bd745d7', 'solution/streamed_head.py': '519d3c101b54d35d9f688b5876f75b2340fc589b66f64e8b9f3ded31c805fa60', 'solution/swiglu.py': 'e665910faeca6091c958ed7a191604484a44216f51d6fa069f5f48220e7472e6', 'solution/test_activation_storage.py': '4a4b8b9819c26745fb10ef8228119981fec75012b140844c43552f0ac42291cf', 'solution/test_cached_head.py': 'da8e1ad8addceb4d8cde5dcaef459ca4da13e95fb07fff4214bf6b1e313ad7fa', 'solution/test_frozen_storage.py': '87c98888cdb260d6e0e3e2c639ab5e9107a26da1708b8b005a2a4aa1d61ef07d', 'solution/test_fusion_cuda.py': '9a97944fcc19bb33b101523c3ce9a79a211d1fd414a48b2af378ace1b09f2786', 'solution/test_head_fused_t4.py': '335d684aa6fdc063d478265209e6ee05fcdf32fa7babe8d3e6c24644679d512d', 'solution/test_head_matmul.py': '1ae409f55282fbd96d3c78845ab22cded87bf453149fbecbbfe7e3f62ec4c9bd', 'solution/test_head_matmul_t4.py': '75931a45b9623e2ceb458904166218d31ffc68ccb08f129da5691cbdc961467d', 'solution/test_integration.py': '924aa09bc683d559630da5623dbf353a6d0caa1221129381158f868108095c65', 'solution/test_kernels.py': 'a45b842a4f22c1aefe404baeff1ac26511253452ecb0f633c985fbc9df8b252a', 'solution/test_lora_dropout.py': '13d096500853e214811ee54dafc8c4c21c74f84e6e5ca0fe6b9a7599d01b055a', 'solution/test_lora_epilogue.py': 'c7394c871558fe12ab684aca5e5872ecd18763467e2ace460ff9016c7efd6421', 'solution/test_nf4.py': '2ee2f46a3011ab76c61675bef3b3e1ce40e0964d1737c92c36b71e8fd05c9a4f', 'solution/test_rmsnorm.py': 'c14a0ac58ec70ffdc392c0a0d4cba3535e36754095130d79329967326110cc1a', 'solution/test_rotary.py': 'a8659bbb8455c6f20868672a7a41e4c96c67b5a066cd443addd0067acf4faf37', 'solution/test_shared_attention_mask.py': '7e9a36143655f96b8b95024643833f57addf4557ca8c759e0f9243a36bedeaa3', 'solution/test_soup_fusion_factory.py': '058daac741c427dd1d39bf91317b86b5bf5b64fc1f8494e14f73b153c4cfdc71', 'solution/test_streamed_head.py': '48c451e73e0623de7a161c37d7e342481f2a12246686b1dd043f02acf77ef436', 'tools/run_t4_readiness.py': '759e422b3098e0dfbe706e0eb3073354378d2620443a8149fee8851bc0cb1678'}
FUSION_PROTECTED = {'harness/_train.py': '45a62642dceab32cf3efcda504de87a9c25ea28b119b1806f56d14ff1146a89d', 'harness/check_rules.py': '879ab1861674f6e714eadd00aaf044235d98526dd75fab1035deec3401121ef5', 'harness/common.py': 'c83066f24f266dab62ebcc9c2d44f5a65902760b53da912e65ebfe511579c2db', 'harness/eval_adapter.py': '47cddbb8e4579c69f9535b099ff2371b1c3af08bfbd93d14bf79aadd74d38482', 'harness/example_reference.py': '6a11055f4a1822e53971c228c6e3e3aa6b88edd9eb6db81a388e26a70a030ceb', 'harness/run_task.py': 'd935a8959f348977313f2eef36870ff2699b9ba013a7978918ff35e148968a79', 'harness/score.py': '731c9b644dcc4069b0b069616f438064d1f7f376a202feb15201fd910ec688c5', 'task/prepare_data.py': 'a8543f11a55aff80dbdc714a0e62391244a5296616eea2e34dc2eb3ff3ec18f7', 'task/task.yaml': 'dd70c739fcd143e06aef3bb1efcc9386f682bf1de26f30e08409d86a97ff47ce', 'tests/conftest.py': 'a82eabab468ea0ef1e003cccd7c3827ce2f4d00d09c32948a34fb48221f7337a', 'tests/test_callback.py': '1919303a3670c0a33abbfce9b33eab0913bb64a3a21c3312d86531c32b5bf490', 'tests/test_config.py': 'db808c44470d144b7cf18b7329492f41f76e37cfc24306e7cba0488bca9d787c', 'tests/test_formats.py': '5d656ed50c59f794e934473d459ea85635f892ba028e192414474e9b5d65919e', 'tests/test_loader.py': '3dad1bea2d33aab97e52b7cd44888d768bc9519af44fbe6cf3e96c260026632f', 'tests/test_trainer_init.py': 'b7608d62a40983bf7c9e5b270e0e527b641a837e462d6df2909fec88f607e821'}
FUSION_PAYLOAD = (
    '...'
)
repo = Path(os.environ["SOUP_REPO"]).resolve(strict=True)
results = Path(os.environ["SOUP_RESULTS"])
results.mkdir(parents=True, exist_ok=True)
for name, expected in FUSION_PROTECTED.items():
    assert hashlib.sha256((repo / name).read_bytes()).hexdigest() == expected, name
payload = base64.b64decode(FUSION_PAYLOAD, validate=True)
assert hashlib.sha256(payload).hexdigest() == FUSION_ARCHIVE_SHA256
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert set(archive.namelist()) == set(FUSION_FILES)
    assert len(archive.namelist()) == len(FUSION_FILES)
    for item in archive.infolist():
        name = PurePosixPath(item.filename)
        assert not name.is_absolute() and ".." not in name.parts
        assert name.parts[0] == "solution" or str(name) == "tools/run_t4_readiness.py"
        assert not stat.S_ISLNK(item.external_attr >> 16)
        raw = archive.read(item)
        assert hashlib.sha256(raw).hexdigest() == FUSION_FILES[str(name)], str(name)
        target = repo.joinpath(*name.parts)
        assert target.resolve().is_relative_to(repo) and not target.is_symlink()
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(raw)

digest = hashlib.sha256()
for path in sorted((repo / "solution").rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        digest.update(str(path.relative_to(repo / "solution")).encode())
        digest.update(path.read_bytes())
assert digest.hexdigest() == FUSION_SOLUTION_SHA256, "Unexpected extra or changed solution source"
for name, expected in FUSION_PROTECTED.items():
    assert hashlib.sha256((repo / name).read_bytes()).hexdigest() == expected, name
(results / "fusion_source_manifest.json").write_text(json.dumps({
    "solution_sha256": FUSION_SOLUTION_SHA256,
    "archive_sha256": FUSION_ARCHIVE_SHA256,
    "files": FUSION_FILES,
    "protected_sources_unchanged": True,
    "service_template_source_commit": '3a4953f76288969bb91956f114eda5b7e541f104',
    "target_seconds": 33,
    "performance_status": "not_measured_by_install_cell"
}, indent=2) + "\n")
print("Installed fused candidate:", FUSION_SOLUTION_SHA256)
print("All optimization source is on disk under", repo / "solution")


In [ ]:
# Fixed research A/B: preserve cached native head and fuse original kernels;
# only enable lossless frozen tied-weight storage.
# This updates ONLY solution/settings.py before physical tests and training.
import hashlib, json
settings_path = repo / "solution" / "settings.py"
old = settings_path.read_text()
assert old.count("FROZEN_STORAGE = False") == 1, "Unexpected candidate source"
assert 'HEAD_MODE = "cached"' in old
assert 'CACHED_CE_CHUNK_TOKENS = 128' in old
new = old.replace("FROZEN_STORAGE = False", "FROZEN_STORAGE = True", 1)
settings_path.write_text(new)
assert settings_path.read_text() == new
digest = hashlib.sha256()
for source_path in sorted((repo / "solution").rglob("*")):
    if source_path.is_file() and "__pycache__" not in source_path.parts:
        digest.update(str(source_path.relative_to(repo / "solution")).encode())
        digest.update(source_path.read_bytes())
final_hash = digest.hexdigest()
record = {
    "experiment": "cached128_frozen_storage",
    "base_embedded_solution_sha256": FUSION_SOLUTION_SHA256,
    "final_solution_sha256": final_hash,
    "modified_only": "solution/settings.py",
    "HEAD_MODE": "cached",
    "CACHED_CE_CHUNK_TOKENS": 128,
    "FROZEN_STORAGE": True,
    "no_harness_or_task_modification": True,
    "source_patched_before_readiness_and_training": True,
}
(results / "fusion_experiment_manifest.json").write_text(json.dumps(record, indent=2)+"\n")
print("Fixed experiment settings / source:", json.dumps(record, indent=2))


In [ ]:
# Run only the solution. The server reuses its saved matching public baseline.
run = Path('/kaggle/working/training-run')
subprocess.run([sys.executable, str(repo / 'harness/run_task.py'), '--out', str(run), '--timeout', '900', '--only', 'solution'], cwd=repo, check=True)


In [ ]:
# Export small reports and logs. Large adapter/checkpoint files remain in this job's VM workspace.
for path in run.rglob('*'):
    if path.is_file() and path.suffix in ('.json', '.jsonl', '.log', '.yaml') and path.stat().st_size < 4 * 1024**2:
        target = results / path.relative_to(run)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(path, target)
print((run / 'result.json').read_text())
# This is a public-data screen. Notebook output does not establish official admission or score.
